# Step 1e — LLM disambiguation (UMLS CUI)

Reads **`grounded_entities.json`** from Step 1d (entity linking). Only rows with `linking_status == "needs_disambiguation"` are sent to the LLM.

**Output:** `resolved_entities.json` in the chosen output directory (same list of entities, with `cui_final` / updated `linking_status`).

**Backends:** `hf_local` (GPU/CPU), `hf_inference` (HF Serverless API), or `none` (copy through without LLM calls).

## 1. Dependencies

In [ ]:
!pip install -q transformers accelerate sentencepiece huggingface_hub
# Local generation (hf_local) — Colab often has torch already; install if import fails
try:
    import torch
except ImportError:
    !pip install -q torch

## 2. Project root and `scripts/` on `sys.path`

- **Google Colab:** this notebook defaults to **`UPLOAD_PROJECT_ZIP = True`** so a **Choose Files** button appears under the next cell — upload a ZIP of your thesis folder that contains `scripts/` (same as the full pipeline). If you already used **Files** in the left sidebar to upload the whole project so `scripts/` exists under `/content`, set `UPLOAD_PROJECT_ZIP = False`.
- **Optional:** set `GIT_CLONE_URL` to a **public** repo URL to clone instead of uploading a ZIP.
- **Local Jupyter:** open this notebook from the thesis repo root, or set `MANUAL_PROJECT_ROOT`, or set `UPLOAD_PROJECT_ZIP = False`.

In [ ]:
import os
import subprocess
import sys
import zipfile
from pathlib import Path
from typing import Optional


def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


IN_COLAB = _in_colab()

# None = auto: True in Google Colab (shows Choose Files), False on local Jupyter
UPLOAD_PROJECT_ZIP: Optional[bool] = None
if UPLOAD_PROJECT_ZIP is None:
    UPLOAD_PROJECT_ZIP = IN_COLAB

# Public git URL only — clone into cwd/thesis_repo if scripts/ not found yet
GIT_CLONE_URL: Optional[str] = None  # e.g. "https://github.com/you/Thesis.git"

# If not using ZIP / clone, set to your repo root, or None to auto-detect
MANUAL_PROJECT_ROOT: Optional[str] = None  # e.g. r"C:\Users\...\Thesis"


def _find_project_with_scripts(here: Path) -> Optional[Path]:
    if (here / "scripts").is_dir():
        return here.resolve()
    for child in sorted(here.iterdir()):
        if child.is_dir() and (child / "scripts").is_dir():
            return child.resolve()
    return None


root = Path.cwd().resolve()

if GIT_CLONE_URL:
    clone_target = root / "thesis_repo"
    if not (clone_target / "scripts").is_dir():
        print(f"Cloning {GIT_CLONE_URL!r} → {clone_target} …")
        subprocess.run(
            ["git", "clone", "--depth", "1", GIT_CLONE_URL, str(clone_target)],
            check=True,
        )
    if (clone_target / "scripts").is_dir():
        os.chdir(clone_target)
        root = Path.cwd().resolve()
        print("Using cloned repo:", root)

if UPLOAD_PROJECT_ZIP:
    try:
        from google.colab import files

        print(
            "Choose Files should appear below — upload your thesis ZIP (must contain scripts/). "
            "If nothing appears, use the Colab menu: Runtime → Run all, or refresh the page."
        )
        uploaded = files.upload()
        for name, data in uploaded.items():
            dest = root / name
            dest.write_bytes(data)
            print(f"Received {len(data) // 1024} KB → {name}")
            if name.lower().endswith(".zip"):
                with zipfile.ZipFile(dest, "r") as zf:
                    zf.extractall(root)
                print("ZIP extracted.")
            break
    except ImportError:
        print("Not in Colab — UPLOAD_PROJECT_ZIP ignored.")

if MANUAL_PROJECT_ROOT:
    proj = Path(MANUAL_PROJECT_ROOT).resolve()
else:
    proj = _find_project_with_scripts(root)

if proj is not None and proj != root:
    os.chdir(proj)
    root = Path.cwd().resolve()
    print(f"Using project folder: {root}")

if not (root / "scripts").is_dir():
    raise RuntimeError(
        "Cannot find scripts/ (need step1e_disambiguate.py, hf_llm.py, utils.py).\n\n"
        "In Colab:\n"
        "  • Re-run this cell with the default UPLOAD_PROJECT_ZIP (True in Colab) and use Choose Files.\n"
        "  • Or upload the project via the left Files sidebar so /content/scripts exists, then set "
        "UPLOAD_PROJECT_ZIP = False and re-run.\n"
        "  • Or set GIT_CLONE_URL to a public repo.\n\n"
        f"(IN_COLAB={IN_COLAB}, UPLOAD_PROJECT_ZIP={UPLOAD_PROJECT_ZIP}, cwd={root})"
    )

scripts_dir = str(root / "scripts")
if scripts_dir not in sys.path:
    sys.path.insert(0, scripts_dir)

print("Working directory:", root)
print("scripts/ on path:", scripts_dir)

## 3. Provide `grounded_entities.json` (Step 1d output)

- **`USE_COLAB_UPLOAD = True`:** use the file picker (upload `grounded_entities.json`).
- **`USE_COLAB_UPLOAD = False`:** set `LOCAL_GROUNDED_JSON` to a path on disk (e.g. `grounded_entities.json` in the repo root).

In [ ]:
from pathlib import Path

USE_COLAB_UPLOAD = True
# When USE_COLAB_UPLOAD is False, path to your Step 1d JSON (relative to cwd or absolute).
LOCAL_GROUNDED_JSON = Path("grounded_entities.json")

out_step1 = root / "outputs" / "step1"
out_step1.mkdir(parents=True, exist_ok=True)
grounded_path = out_step1 / "grounded_entities.json"

if USE_COLAB_UPLOAD:
    try:
        from google.colab import files

        print("Upload grounded_entities.json (output of Step 1d).")
        up = files.upload()
        if not up:
            raise RuntimeError("No file uploaded.")
        name, data = next(iter(up.items()))
        grounded_path.write_bytes(data)
        print(f"Saved {len(data) // 1024} KB → {grounded_path}")
    except ImportError:
        print("Not in Colab — using LOCAL_GROUNDED_JSON.")
        src = LOCAL_GROUNDED_JSON if LOCAL_GROUNDED_JSON.is_absolute() else root / LOCAL_GROUNDED_JSON
        if not src.is_file():
            raise FileNotFoundError(f"Missing: {src}")
        grounded_path.write_bytes(src.read_bytes())
        print(f"Copied {src} → {grounded_path}")
else:
    src = LOCAL_GROUNDED_JSON if LOCAL_GROUNDED_JSON.is_absolute() else root / LOCAL_GROUNDED_JSON
    if not src.is_file():
        raise FileNotFoundError(f"Missing: {src}")
    grounded_path.write_bytes(src.read_bytes())
    print(f"Copied {src} → {grounded_path}")

print("Input for disambiguate():", grounded_path)

## 4. LLM settings — Hugging Face token

Create a token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) (read access is enough for downloads; gated models need you to accept the model license on the model page).

**Ways to provide it (first match wins):**

1. **`HF_TOKEN_INLINE`** in the next cell — paste your token inside the quotes, e.g. `HF_TOKEN_INLINE = "hf_xxxx"`.
2. **Colab Secrets:** key icon → add secret **`HF_TOKEN`** → turn **Notebook access** ON for that secret (otherwise `userdata.get` cannot read it).
3. **Environment:** `HF_TOKEN` already set in the runtime.
4. If still missing, the next cell **asks you to paste the token** when you run it (Colab shows an input box under the cell).

The cell also runs `huggingface_hub.login(...)` so gated Llama weights can download from the Hub.

In [ ]:
import os

HF_MODEL = os.environ.get("HF_MODEL", "meta-llama/Llama-3.2-3B-Instruct")

# Paste between quotes, e.g. HF_TOKEN_INLINE = "hf_xxxxxxxx"
HF_TOKEN_INLINE = ""

# If True, when no token was found you will be prompted to paste once (works in Colab).
PROMPT_FOR_TOKEN_IF_MISSING = True


def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


HF_TOKEN = (HF_TOKEN_INLINE or "").strip() or None
if not HF_TOKEN and _in_colab():
    try:
        from google.colab import userdata

        raw = userdata.get("HF_TOKEN")
        HF_TOKEN = (raw or "").strip() or None
    except Exception as exc:
        print(
            "Colab: could not read secret HF_TOKEN (create it and enable Notebook access):",
            type(exc).__name__,
            str(exc) or repr(exc),
        )

if not HF_TOKEN:
    HF_TOKEN = (os.environ.get("HF_TOKEN") or "").strip() or None

if not HF_TOKEN and PROMPT_FOR_TOKEN_IF_MISSING:
    try:
        t = input(
            "Paste your Hugging Face token (hf_...) and press Enter, or Enter alone to skip:\n"
        ).strip()
        HF_TOKEN = t or None
    except EOFError:
        pass

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    try:
        from huggingface_hub import login

        login(token=HF_TOKEN, add_to_git_credential=False)
        print("Hugging Face: login OK (session authenticated).")
    except Exception as exc:
        print("Hugging Face: login() issue (step may still work if token is passed explicitly):", exc)
else:
    print(
        "WARNING: No HF_TOKEN — Step 1e needs it for gated Llama on the Hub.\n"
        "Fix: set HF_TOKEN_INLINE = \"hf_...\" in this cell, or Colab Secret HF_TOKEN with Notebook access ON, "
        "or run again and paste when prompted."
    )

print("HF_MODEL:", HF_MODEL)
print("HF_TOKEN set:", bool(HF_TOKEN))

## 5. Run disambiguation

In [ ]:
from step1e_disambiguate import disambiguate

resolved = disambiguate(
    grounded_path=str(grounded_path),
    output_dir=str(out_step1),
    hf_token=HF_TOKEN,
    hf_model=HF_MODEL,
)

linked = sum(1 for e in resolved if e.get("cui_final"))
need = sum(1 for e in resolved if e.get("linking_status") == "needs_disambiguation")
print(f"\n→ {linked}/{len(resolved)} entities have a final CUI")
print(f"→ {need} still need disambiguation (if LLM errors occurred)")
print("Written:", out_step1 / "resolved_entities.json")

## 6. (Colab) Download `resolved_entities.json`

In [ ]:
try:
    from google.colab import files

    dl = out_step1 / "resolved_entities.json"
    if dl.is_file():
        files.download(str(dl))
        print("Download started for resolved_entities.json")
    else:
        print("No resolved_entities.json found.")
except ImportError:
    print("Not in Colab — open the file locally:", out_step1 / "resolved_entities.json")